# Topic 1: Uncertainty-Triggered RAG Across Decoders
**Author:** Khushi Shukla  
**Target:** ARR Deadline (4 Jan 2027)  

This notebook runs the official ARR evaluation pipeline:
1. **Step 2:** Hashimoto et al. TriviaQA greedy baseline (Rouge-L, EM, PRR x100)
2. **Step 4:** Multi-decoder dynamic RAG evaluation:
   - **Table 1 (RQ1a):** Fixed-threshold retrieval rate shift
   - **Table 2 (RQ1b):** Matched-retrieval-rate Answer F1 & EM
   - **Table 3 (RQ1c):** Post-retrieval context copying rate (Greedy vs Contrastive)

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn scipy

In [ ]:
# 2. Authenticate Hugging Face Token
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        from huggingface_hub import login
        login(token)
        print("Hugging Face authenticated successfully.")
except Exception as e:
    print("No Colab secret HF_TOKEN found; proceeding with unauthenticated public access.")

In [ ]:
# 3. Setup Repository (Clean single directory: clone if new, pull if existing)
import os
if not os.path.exists("/content/token_retrieval"):
    !git clone https://github.com/Khushi281300/token_retrieval.git /content/token_retrieval
%cd /content/token_retrieval
!git pull origin main

In [ ]:
# 4. Step 2: Hashimoto et al. Greedy Baseline (Rouge-L, EM, PRR x100)
!python scripts/reproduce_hashimoto.py --model allenai/OLMo-2-1124-7B-Instruct --samples 100 --unit nats

In [ ]:
# 5. Step 4: Full Multi-Decoder Evaluation (Table 1, Table 2, Table 3)
# - Primary: Answer Decoder = GREEDY (Draft varies: Greedy, Beam 5, Contrastive alpha=0.6, Nucleus 3 seeds)
# - Secondary: Draft = GREEDY (Answer varies: Greedy vs Contrastive)
!python scripts/run_experiments.py --model allenai/OLMo-2-1124-7B-Instruct --samples 100 --target_rate 0.40 --score_type raw